# Interpretability comparison: decision path against relational rule

A qualitative companion to `propositional_baseline.ipynb`, requested during review.

Both models can be read by a human, but they say different kinds of things. This notebook extracts the root-to-leaf path by which the decision tree predicts the MRS medium, so that it can be placed beside a relational rule mined for the same medium (Listing 1.3 of the manuscript) and the two forms of explanation compared.

MRS medium is used because it is one of the two examples highlighted in the manuscript, and the only one available here: Medium 119 does not pass the minimum class support, so the tree cannot predict it. Note that the path below is the tree's route to MRS in general; how the tree treats the individual organism of the manuscript's example is a separate question, examined at the end.

## Setup

In [1]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

DATA = "../data"
RANDOM_SEED = 12
MAX_DEPTH = 4
TARGET = "medium:11"

## Data and model

The same tree as in `propositional_baseline.ipynb`: identical splits, target definition, depth and seed.

In [2]:
EXCLUDED_MEDIA = [
    "medium:104", "medium:1a", "medium:215", "medium:381", "medium:545",
    "medium:553", "medium:830c", "medium:J12", "medium:514c",
]

nodes = pd.read_csv(f"{DATA}/merged-kg_nodes.tsv", sep="\t", usecols=["id", "name"], dtype=str)
edges = pd.read_csv(f"{DATA}/merged-kg_edges.tsv", sep="\t", usecols=["subject", "object"], dtype=str)

names = nodes.set_index("id")["name"]
agar_media = set(names[names.str.contains("agar", case=False, na=False)].index)

organism_medium_edges = edges[
    edges["subject"].str.startswith("NCBITaxon:", na=False)
    & edges["object"].str.startswith("medium:", na=False)
].drop_duplicates()

organism_medium_edges = organism_medium_edges[
    ~organism_medium_edges["object"].isin(agar_media)
    & ~organism_medium_edges["object"].isin(EXCLUDED_MEDIA)
]

media_per_organism = organism_medium_edges.groupby("subject")["object"].agg(set)
targets = media_per_organism[media_per_organism.apply(len) == 1].apply(min)

X_train = pd.read_csv(f"{DATA}/catboost_train.tsv", sep="\t", index_col="subject")
y_train = targets.reindex(X_train.index)

tree = DecisionTreeClassifier(max_depth=MAX_DEPTH, random_state=RANDOM_SEED).fit(X_train, y_train)
print(f"tree fitted: {tree.get_n_leaves()} leaves, depth {tree.get_depth()}")

tree fitted: 15 leaves, depth 4


## The decision path

Every node is walked back from the leaf that predicts MRS medium to the root, recording which side of each split was taken.

In [3]:
structure = tree.tree_

parent = {}
for node in range(structure.node_count):
    for child, is_left in [(structure.children_left[node], True),
                           (structure.children_right[node], False)]:
        if child != -1:
            parent[child] = (node, is_left)

leaves = [n for n in range(structure.node_count) if structure.children_left[n] == -1]
target_leaf = next(n for n in leaves if tree.classes_[structure.value[n].argmax()] == TARGET)

path = []
node = target_leaf
while node in parent:
    node, is_left = parent[node]
    path.append((X_train.columns[structure.feature[node]], not is_left))
path.reverse()

for position, (feature, present) in enumerate(path):
    keyword = "IF " if position == 0 else "AND"
    state = "    " if present else "NOT "
    print(f"{keyword} {state}{feature:24} {names.get(feature, '')}")
print(f"THEN    {TARGET:24} {names.get(TARGET, '')}")

samples = structure.n_node_samples[target_leaf]
share = structure.value[target_leaf].ravel()[list(tree.classes_).index(TARGET)]
print(f"\ntraining samples stored at this leaf: {samples}, "
      f"of which {TARGET}: {round(samples * share)} ({share:.0%})")

IF  NOT gram_stain:negative      gram negative
AND     oxygen:microaerophile    microaerophile
AND     CHEBI:59640              N-acetylglucosamine
AND NOT EC:3.1.3.1               alkaline phosphatase
THEN    medium:11                MRS MEDIUM (DSMZ Medium 11)

training samples stored at this leaf: 43, of which medium:11: 40 (93%)


## Side by side

The path above, against the relational rule mined for the same medium (Listing 1.3 of the manuscript):

```
( ?a biolink:location_of  isolation_source:food_fermented )
( ?a biolink:capable_of   oxygen:microaerophile )
( ?a biolink:has_phenotype temperature:mesophilic )
    => ( ?a biolink:occurs_in medium:11 )

Support: 41; PcaConfidence: 0.84; Lift: 103.38
```

Both carry three conditions, and both use `oxygen:microaerophile`. The rule states typed relationships between the organism and named biological entities, and reports the evidence supporting it. The tree states presence and absence of flattened binary columns, and its leaf carries a bare class label. The two are compared in the manuscript.

## The manuscript's example organism

The relational rules predict MRS medium for *Ligilactobacillus acidipiscis*. The graph holds three nodes for it — the species and two strains — which the flattened representation treats as unrelated rows. Their feature vectors are rebuilt from the graph, using the columns the model was trained on, and passed to the tree.

In [4]:
LIGILACTOBACILLUS = ["NCBITaxon:89059", "NCBITaxon:1423716", "NCBITaxon:1046599"]

def feature_row(organism):
    linked = set(edges.loc[edges["subject"] == organism, "object"]) | set(
        edges.loc[edges["object"] == organism, "subject"])
    return [1 if column in linked else 0 for column in X_train.columns]

rows = pd.DataFrame([feature_row(o) for o in LIGILACTOBACILLUS],
                    columns=X_train.columns, index=LIGILACTOBACILLUS)

for organism, prediction in zip(LIGILACTOBACILLUS, tree.predict(rows)):
    traits = int(rows.loc[organism].sum())
    print(f"{names[organism]:42} {traits:3} traits -> {prediction} {names.get(prediction, '')}")

Ligilactobacillus acidipiscis               35 traits -> medium:693 COLUMBIA BLOOD MEDIUM (DSMZ Medium 693)
Ligilactobacillus acidipiscis DSM 15836     15 traits -> medium:11 MRS MEDIUM (DSMZ Medium 11)
Ligilactobacillus acidipiscis KCTC 13900     2 traits -> medium:514 BACTO MARINE BROTH (DIFCO 2216) (DSMZ Medium 514)


The path the species node actually takes through the same tree, read off with `decision_path`:

In [5]:
def show_path(organism):
    for position, node in enumerate(tree.decision_path(rows.loc[[organism]]).indices):
        if structure.children_left[node] == -1:
            medium = tree.classes_[structure.value[node].argmax()]
            print(f"THEN    {medium:24} {names.get(medium, '')}")
        else:
            feature = X_train.columns[structure.feature[node]]
            present = rows.loc[organism, feature] == 1
            keyword = "IF " if position == 0 else "AND"
            print(f"{keyword} {'    ' if present else 'NOT '}{feature:24} {names.get(feature, '')}")


show_path("NCBITaxon:89059")

IF  NOT gram_stain:negative      gram negative
AND     oxygen:microaerophile    microaerophile
AND     CHEBI:59640              N-acetylglucosamine
AND     EC:3.1.3.1               alkaline phosphatase
THEN    medium:693               COLUMBIA BLOOD MEDIUM (DSMZ Medium 693)


The gram stain of the species node is not curated either way, so both `gram_stain:negative` and `gram_stain:positive` are zero:

In [6]:
rows.loc[LIGILACTOBACILLUS, ["gram_stain:negative", "gram_stain:positive",
                             "oxygen:microaerophile", "CHEBI:59640", "EC:3.1.3.1"]]

,gram_stain:negative,gram_stain:positive,oxygen:microaerophile,CHEBI:59640,EC:3.1.3.1
NCBITaxon:89059,0,0,1,1,1
NCBITaxon:1423716,0,0,1,1,0
NCBITaxon:1046599,0,0,0,0,0
